# Deliverable 3 - INF201 

Submission deadline: 13th Nov 2026

### Hand-in information:
* This is a group submission, so one person submits on behalf of the pair/group
* Submit Python scripts (.py), a Jupyter notebook (ipynb) <i>and</i> a pdf of the same notebook
    * See Overview and objectives for a list of what your notebook should contain
* Include a link to your GitHub repository and a log of the commits
    * To generate the log, paste this into the terminal after your last commit:<br>
    `git log --graph --pretty=format:"%h - %an, %ar : %s" > git_log.txt`
    * The repository should contain a `requirements.txt` with information on your virtual environment
    * Note: Remember to create a new virtual environment for each deliverable
* Include a discussion of GenAI use, for example a summary of the chat log as you have
done earlier
    * This can be a separate file or be included in the Jupyter notebook
* Make a short (5 mins) video discussing your code and results
    * Note that you both have to contribute to the video
    * Either by recording it together
    * Or recording parts separately and editing it together
* We would appriciate it if use English for this assignment, both in the writing and in the video

## Overview and objectives

In this assignment you will continue working on the Character class in `game_simulator_v1.py` that we created in lecture_08 (check out the Week 44 module), and expand this to create a mini-RPG in the form of a battle simulator. To do this you will update and extend the existing class and methods, introduce new classes allowing us to use dice and weapons, and create a combat manager class.

The rules for the game is as follows:
* The game you build should allow two characters to battle until one of them wins. 
* The characters can use weapons to attack, where the damage dealt is determined by a dice roll.
* The characters can have different specialisations (warrior or mage). 
* A character's HP (hit points, or health) must be between 0 and the max given for this specific character.
* The character dies when it reaches 0 HP.

Learning Objectives:
* Enforce data encapsulation using private attributes and properties.
* Implement custom classes to model discrete concepts (`Dice`, `Weapon`, `Battle`).¨
* Create subclasses inheriting from the `Character` class.
* Apply object composition by equipping characters with weapons and dice.

To organise your project, we recommend the following struckture:
* Separate modules (Python scripts) for each of the classes created
* A Jupyter Notebook containing the following:
    * A presentation of each class created (showing attributes and methods objects of the class has)
    * Simulated battles between different types of characters, with and without weapons

   ## Setup
   The classes are written in separate Python modules (`.py` files) and imported here. All imports are collected in this cell, so the notebook can be run from top to bottom.

In [1]:
%load_ext autoreload
%autoreload 2

import random
from collections import Counter

from character import Character
from dice import Dice

## Task 1: Update the Character class

When we created the Character class in the lecture we first included an `attack()` method, and afterwards indicated that we wanted hit points to be a protected feature `_hp`. This lead to the `Character.attack()` method changing the target's protected attribute. Modifying another object's private attribute is bad form! Your first task is to fix this.

* Implement a `take_damage()` method to be used when the `Character` object is injured. This method should only affect the characters *own* HP.
* Implement a `heal()` method to be used when the `Character` heals (in other words increase their hit points)
* Change the existing `attack()` method so that calls the target's `take_damage()` method rather than directly modifying the target's `_hp`.
* Add an `is_alive` property to the `Character` class. This should be read-only, and return True if `_hp`$ > 0$ and False otherwise.

### Task 1 – Changes to the Character class
* `take_damage(amount)` reduces the character's **own** `_hp`. `max(..., 0)` makes sure HP never goes below 0. A negative amount is set to 0, so taking damage can never heal the character.
* `heal(amount)` increases the character's own `_hp`. `min(..., hp_max)` makes sure HP never goes above the maximum. A negative amount is set to 0, so healing can never hurt.
* `attack(target)` now calls `target.take_damage(self.damage)` instead of changing `target._hp` directly. Each character is responsible for its own HP, which is the idea behind encapsulation.
* `is_alive` is a read-only property that returns `True` if `_hp > 0`. It has no setter, so it cannot be changed from outside the class.

Together, `take_damage()` and `heal()` make sure HP always stays between 0 and `hp_max`, as the rules of the game require.

In [2]:
hero = Character("Hero")
enemy = Character("Goblin", hp_max=25, damage=5)

print(hero)
print(enemy)
print(repr(hero))

# Attributes and methods of a Character object (special methods like __init__ are left out)
print("\nAttributes and methods:", [name for name in dir(hero) if not name.startswith("__")])

# The hero attacks until the goblin is dead
print()
while enemy.is_alive:
    hero.attack(enemy)
    print(f"{hero.name} attacks {enemy.name}: {enemy.name} has {enemy.hp} HP, alive: {enemy.is_alive}")

# Healing cannot go above hp_max
hero.take_damage(30)
print(f"\n{hero.name} takes 30 damage: {hero.hp} HP")
hero.heal(50)
print(f"{hero.name} heals 50: {hero.hp} HP (max is {hero.hp_max})")

Character: Hero with 100 HP and 10 damage
Character: Goblin with 25 HP and 5 damage
<Character name=Hero hp=100 damage=10>

Attributes and methods: ['_hp', 'attack', 'damage', 'heal', 'hp', 'hp_max', 'introduce', 'is_alive', 'name', 'take_damage']

Hero attacks Goblin: Goblin has 15 HP, alive: True
Hero attacks Goblin: Goblin has 5 HP, alive: True
Hero attacks Goblin: Goblin has 0 HP, alive: False

Hero takes 30 damage: 70 HP
Hero heals 50: 100 HP (max is 100)


### Task 1 – Tests
The tests check that HP always stays between 0 and `hp_max`, that `attack()` only changes the target's HP through `take_damage()`, and that `hp` and `is_alive` are read-only.

In [3]:
import inspect

# take_damage: HP goes down, but never below 0, and negative damage does not heal
c = Character("Test", hp_max=50)
c.take_damage(20)
assert c.hp == 30
c.take_damage(100)
assert c.hp == 0 and not c.is_alive
c.take_damage(-5)
assert c.hp == 0

# heal: HP goes up, but never above hp_max, and negative healing does not hurt
c = Character("Test", hp_max=50)
c.take_damage(30)
c.heal(10)
assert c.hp == 30
c.heal(100)
assert c.hp == 50
c.heal(-10)
assert c.hp == 50

# attack: only the target loses HP, and attack() does not touch the target's _hp directly
attacker = Character("Attacker", damage=15)
target = Character("Target")
attacker.attack(target)
assert target.hp == 85 and attacker.hp == 100
assert "_hp" not in inspect.getsource(Character.attack)

# is_alive: True while HP > 0, False at 0
c = Character("Test", hp_max=10)
assert c.is_alive
c.take_damage(10)
assert not c.is_alive

# hp and is_alive are read-only
for attribute in ["hp", "is_alive"]:
    try:
        setattr(c, attribute, 100)
        raise AssertionError(f"{attribute} should be read-only")
    except AttributeError:
        pass

print("All Task 1 tests passed.")

All Task 1 tests passed.


## Task 2: Introducing randomness with a Dice class

Currently, characters deal fixed damage. To introduce variable damage, create a separate `Dice` class.

* Create a `Dice` class that can create any n-sided Dice (n>=2), where the default number of sides is 6.
* Add a `roll()` method to roll the dice, using `random.randint(1, self.sides)`.
* Add a `__repr__` method that returns string representations like `Dice(sides=6)`.
* Implement a `__call__` method that allow you to roll the dice created by `d20 = Dice(sides=20)` by calling `d20.roll()`.
    * See the [Python documentation of `__call__`](https://docs.python.org/3/reference/datamodel.html#object.__call__) or [this example](https://www.w3schools.com/python/python_magic_call.asp)

### Task 2 – The Dice class
The `Dice` class is written in its own module, `dice.py`.
* `Dice(sides=6)` creates a dice with any number of sides. The default is 6. The number of sides is checked when the dice is created: fewer than 2 sides raises a `ValueError`, and a non-integer (for example `6.5` or `"6"`) raises a `TypeError`. This way an invalid dice can never exist.
* The number of sides is stored in the protected attribute `_sides`, and shown through the read-only property `sides`. This follows the same idea as `hp` in the `Character` class: the value can be read, but not changed to something invalid after the dice is created.
* `roll()` returns `random.randint(1, self.sides)`, which is a random integer from 1 to `sides`, where both ends are included.
* `__repr__` returns a string like `Dice(sides=6)`. This is also valid Python code that recreates the same dice.
* `__call__` makes the object callable, so `d20()` rolls the dice. It simply returns `self.roll()`, so `d20()` and `d20.roll()` always do the same thing. (The task text writes `d20.roll()`, but it is `__call__` that makes the shorter `d20()` possible.)

In the demo below we set a fixed random seed with `random.seed()`, so the notebook gives the same output every time it is run.

In [4]:
random.seed(201)  # Fixed seed, so the output is the same every time the notebook is run

d6 = Dice()           # Default: 6 sides
d20 = Dice(sides=20)

print(d6)             # print() falls back to __repr__ because Dice has no __str__
print(repr(d20))
print(f"d6 has {d6.sides} sides and d20 has {d20.sides} sides")

# Attributes and methods of a Dice object (special methods like __init__ are left out)
print("\nAttributes and methods:", [name for name in dir(d20) if not name.startswith("__")])

# roll() and __call__ do the same thing
print("\nFive rolls with d20.roll():", [d20.roll() for _ in range(5)])
print("Five rolls with d20():     ", [d20() for _ in range(5)])

# Roll the six-sided dice many times to see that all sides come up about equally often
rolls = Counter(d6() for _ in range(6000))
print("\n6000 rolls with d6 (expected about 1000 of each):")
for side in range(1, d6.sides + 1):
    print(f"  {side}: {rolls[side]}")

# Trying to create a dice with only one side
try:
    Dice(sides=1)
except ValueError as error:
    print(f"\nDice(sides=1) -> ValueError: {error}")

Dice(sides=6)
Dice(sides=20)
d6 has 6 sides and d20 has 20 sides

Attributes and methods: ['_sides', 'roll', 'sides']

Five rolls with d20.roll(): [3, 11, 14, 1, 10]
Five rolls with d20():      [11, 16, 20, 9, 15]

6000 rolls with d6 (expected about 1000 of each):
  1: 1032
  2: 983
  3: 953
  4: 1051
  5: 1004
  6: 977

Dice(sides=1) -> ValueError: A dice must have at least 2 sides, got 1


### Task 2 – Tests
The tests check the default number of sides, the `__repr__` string, that every roll is between 1 and `sides`, that all sides can come up, that `d20()` gives the same results as `d20.roll()` with the same seed, that invalid dice cannot be created, and that `sides` is read-only.

In [5]:
# Default dice has 6 sides, and any number of sides >= 2 is allowed
assert Dice().sides == 6
assert Dice(sides=2).sides == 2
assert Dice(sides=100).sides == 100

# __repr__ gives strings like Dice(sides=6)
assert repr(Dice()) == "Dice(sides=6)"
assert repr(Dice(sides=20)) == "Dice(sides=20)"

# roll() and __call__ always return an integer from 1 to sides
for sides in [2, 6, 20]:
    dice = Dice(sides)
    for _ in range(1000):
        roll = dice.roll()
        assert isinstance(roll, int) and 1 <= roll <= sides
        assert 1 <= dice() <= sides

# All sides come up when rolling many times
d6 = Dice()
assert {d6() for _ in range(1000)} == {1, 2, 3, 4, 5, 6}

# d20() and d20.roll() give exactly the same results with the same seed
d20 = Dice(sides=20)
random.seed(42)
rolls_with_roll = [d20.roll() for _ in range(20)]
random.seed(42)
rolls_with_call = [d20() for _ in range(20)]
assert rolls_with_roll == rolls_with_call

# A dice with fewer than 2 sides cannot be created
for invalid_sides in [1, 0, -6]:
    try:
        Dice(invalid_sides)
        raise AssertionError(f"Dice({invalid_sides}) should raise ValueError")
    except ValueError:
        pass

# The number of sides must be an integer
for invalid_sides in [6.5, "6"]:
    try:
        Dice(invalid_sides)
        raise AssertionError(f"Dice({invalid_sides!r}) should raise TypeError")
    except TypeError:
        pass

# sides is read-only
try:
    d6.sides = 1
    raise AssertionError("sides should be read-only")
except AttributeError:
    pass

print("All Task 2 tests passed.")

All Task 2 tests passed.


## Task 3: Create items

Create a `Weapon` class in `items.py` that utilizes your `Dice` class for damage calculations. We call the module `items.py` rather than `weapons.py` so that it can be expanded to include other items your characters can hold.

* Define the class `Weapon` with the attributes `name` and `damage_dice: Dice`.
* Include a `roll_damage()` method that calls `self.damage_dice.roll()`
* Update `Character.__init__` to accept an optional weapon.
* If a character has an equipped weapon, `attack()` should roll damage from the weapon instead of using default base damage.

## Task 4: Character specialization

Extend the character module to include two specialised subclasses of `Character`.

1. `Warrior` subclass (defensive specialist)
    * New attribute: `armor: int`, with a default value of 5
    * Override the `Character.take_damage()` method by reducing the incoming damage by armor before applying it.
2. `Mage` subclass (resource-based attacker)
    * New attributes: `mana: int` and `mana_max: int`, both with default value 30
    * Magic attacks consume mana, so the mana available determines whether the mage can use magic.
    * Override the `Character.attack()` method:
        * If the mage has at least 10 mana left, spend 10 mana to deal a magical spell attack dealing double damage. This magic attack does not use any weapons, even if the character has one equipped.
        * If the mage has less than 10 mana left, fall back to the base attack, and restore 5 mana. This base attack should use whichever weapon the character has equipped.

## Task 5: Running battles

Create a `Battle` class to manage turn-based combat between two `Character` objects.

* Initialize `Battle` with `hero: Character` and `enemy: Character`.
* Implement a `play_round()` method where `hero` attacks `enemy`, and if `enemy` is still alive, `enemy` attacks `hero`.
* Implement a `simulate_battle()` method with a `while` loop that runs rounds until one character is no longer alive.